# VLMEvalKit: Evaluate Qwen2-VL-2B-Instruct on Google Colab

This notebook runs evaluation of the **Qwen2-VL-2B-Instruct** model using [VLMEvalKit](https://github.com/open-compass/VLMEvalKit).

**Requirements:** A Colab GPU runtime (T4 is sufficient for the 2B model).

Go to **Runtime → Change runtime type → GPU (T4)** before running.

## 1. Install Dependencies

In [ ]:
# Install core dependencies (Colab already has PyTorch with CUDA)
!pip install transformers accelerate qwen-vl-utils -q

# Install flash-attn from a pre-built wheel to avoid compilation.
# If no matching wheel is available, we skip it — evaluation still works fine.
import sys, subprocess, torch, urllib.request, urllib.error

def install_flash_attn():
    # Use the CUDA version PyTorch was compiled against (not the system CUDA driver)
    cuda = torch.version.cuda.replace('.', '')               # e.g. "128" or "124"
    torch_v = torch.__version__.split('+')[0].replace('.', '') # e.g. "251"
    py = f"cp{sys.version_info.major}{sys.version_info.minor}" # e.g. "cp312"

    # flash-attention pre-built wheels are published for specific CUDA versions.
    # If the exact version isn't available, try cu124 (latest commonly published).
    candidates = [cuda]
    if cuda not in ("121", "122", "123", "124"):
        candidates.append("124")

    base = "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4"
    for cu in candidates:
        url = f"{base}/flash_attn-2.7.4+cu{cu}torch{torch_v}cxx11abiFALSE-{py}-{py}-linux_x86_64.whl"
        # Probe the URL before attempting install — avoids triggering a source build on 404
        try:
            urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=10)
        except urllib.error.HTTPError:
            continue  # wheel doesn't exist for this CUDA/torch combination
        except Exception as e:
            print(f"Network check failed ({e}); skipping flash-attn.")
            return

        print(f"Installing pre-built flash-attn (cu{cu}, torch{torch_v}, {py})...")
        r = subprocess.run([sys.executable, "-m", "pip", "install", url, "--no-deps", "-q"],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print("flash-attn installed successfully.")
        else:
            print("Install failed:", r.stderr.strip())
        return  # stop after first successful URL probe

    print(f"No pre-built flash-attn wheel for CUDA {torch.version.cuda} / PyTorch {torch.__version__}.")
    print("Skipping flash-attn — evaluation still works (standard attention will be used).")

install_flash_attn()

In [ ]:
# Install VLMEvalKit
!pip install git+https://github.com/open-compass/VLMEvalKit.git -q

## 2. Verify GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 3. Run Evaluation

We evaluate on **MMBench_V11_MINI** (a small subset) for a quick demo. You can change the dataset to any supported benchmark.

Some commonly used datasets:
- `MMBench_DEV_EN` — MMBench English dev set
- `MMStar` — MMStar benchmark
- `MME` — MME benchmark
- `MMMU_DEV_VAL` — MMMU dev+val
- `SEEDBench_IMG` — SEED-Bench image

Use the MINI variants (`MMBench_V11_MINI`, `MMStar_MINI`) for quick testing.

In [ ]:
# Configuration — edit these as needed
MODEL_NAME = "Qwen2-VL-2B-Instruct"
DATASET = "MMBench_V11_MINI"  # Use MINI variant for quick testing
WORK_DIR = "/content/outputs"

In [ ]:
!python -m vlmeval.tools run \
    --data {DATASET} \
    --model {MODEL_NAME} \
    --work-dir {WORK_DIR}

## 4. View Results

In [ ]:
import os
import glob
import pandas as pd

# Find result files
result_files = glob.glob(os.path.join(WORK_DIR, "**", "*result*.csv"), recursive=True)
result_files += glob.glob(os.path.join(WORK_DIR, "**", "*result*.xlsx"), recursive=True)

if result_files:
    for f in sorted(result_files):
        print(f"\n--- {os.path.basename(f)} ---")
        if f.endswith(".csv"):
            df = pd.read_csv(f)
        else:
            df = pd.read_excel(f)
        display(df)
else:
    print("No result files found yet. Check the work directory:")
    !find {WORK_DIR} -type f | head -20

## 5. (Optional) Run on Additional Datasets

In [ ]:
# Uncomment to run on more datasets:

# !python -m vlmeval.tools run --data MMStar_MINI --model Qwen2-VL-2B-Instruct --work-dir /content/outputs
# !python -m vlmeval.tools run --data MME --model Qwen2-VL-2B-Instruct --work-dir /content/outputs
# !python -m vlmeval.tools run --data SEEDBench_IMG --model Qwen2-VL-2B-Instruct --work-dir /content/outputs